<a href="https://colab.research.google.com/github/abrarhossam51-max/Phantoms-AI-W6/blob/main/W6D5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

import os
import json
import time
from itertools import product
from groq import Groq

# Use the same API key from W6D1
client = Groq(api_key=os.environ.get("GROQ_API_KEY"))

MODEL = "openai/gpt-oss-20b"
TEMPERATURE = 0.2
RUNS = 3

reviews = [
    {
        "id": 1,
        "text": "اشتريت سماعة Anker من أحمد في المعادي، الصوت ممتاز والبطارية بتعدي اليوم كله. التوصيل جه في معاده. أنصح بيها."
    },
    {
        "id": 2,
        "text": "موبايل Samsung A54 كاميرته حلوة جداً بس البطارية بتخلص قبل نص اليوم والشاحن بطيء. مش عارف أنصح بيه ولا لأ."
    },
    {
        "id": 3,
        "text": "التوصيل اتأخر أسبوع كامل ومحدش رد على الرسايل. الخدمة وحشة جداً."
    },
    {
        "id": 4,
        "text": "برافو عليهم، طلبت الشنطة تيجي في يومين فجت بعد تلات أسابيع، وكمان الزرار اتكسر من أول استخدام. خدمة تحفة فعلاً 👏"
    },
    {
        "id": 5,
        "text": "لابتوب Lenovo IdeaPad اللي استلمته من فرع سيتي ستارز شغال تمام والسعر مناسب، بس مفيش كرتونة أصلية والتغليف كان ضعيف."
    },
]

# Keep the original prompts exactly as required
original_prompts = [
    "قولي رأيك في الريفيو ده وهل هو كويس ولا وحش، ومتطولش، بس اشرح ليه، وطلعلي أي أسماء أو منتجات مذكورة. الريفيو: " + reviews[0]["text"],
    "قولي رأيك في الريفيو ده وهل هو كويس ولا وحش، ومتطولش، بس اشرح ليه، وطلعلي أي أسماء أو منتجات مذكورة. الريفيو: " + reviews[1]["text"],
    "قولي رأيك في الريفيو ده وهل هو كويس ولا وحش، ومتطولش، بس اشرح ليه، وطلعلي أي أسماء أو منتجات مذكورة. الريفيو: " + reviews[2]["text"],
    "قولي رأيك في الريفيو ده وهل هو كويس ولا وحش، ومتطولش، بس اشرح ليه، وطلعلي أي أسماء أو منتجات مذكورة. الريفيو: " + reviews[3]["text"],
    "قولي رأيك في الريفيو ده وهل هو كويس ولا وحش، ومتطولش، بس اشرح ليه، وطلعلي أي أسماء أو منتجات مذكورة. الريفيو: " + reviews[4]["text"],
]

# The repaired prompt uses a fixed system instruction and output schema
system_prompt = """
You are a review-analysis assistant.
Follow the user's instructions exactly.
Return one valid JSON object only.
Do not use Markdown fences or add text outside the JSON.
Use the same keys and value types for every review.
"""

def make_fixed_prompt(review):
    return f"""
Context:
The following text is a customer review in Egyptian Arabic:
{review}

Instruction:
1. Decide the overall sentiment of the review.
2. Use only one sentiment label:
   "positive", "negative", or "mixed".
3. Explain the decision briefly using evidence from the review.
4. Extract names, brands, products, and locations explicitly mentioned.
5. Do not invent missing information.
6. If no entities of a type are mentioned, use an empty list.
7. If a detail cannot be determined, use null.

Output Format:
Return exactly this JSON structure:
{{
  "sentiment": "positive, negative, or mixed",
  "explanation": "short explanation in Arabic",
  "entities": [
    {{
      "text": "exact name or product mentioned",
      "type": "person, brand, product, or location"
    }}
  ]
}}
"""


def ask_model(user_prompt, use_fixed_prompt=False):
    if use_fixed_prompt:
        response = client.chat.completions.create(
            model=MODEL,
            temperature=TEMPERATURE,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt},
            ],
        )
    else:
        response = client.chat.completions.create(
            model=MODEL,
            temperature=TEMPERATURE,
            messages=[
                {"role": "user", "content": user_prompt},
            ],
        )

    return response.choices[0].message.content


def parse_json_response(response):
    """Return True only when the complete response is valid JSON."""
    try:
        json.loads(response)
        return True
    except (json.JSONDecodeError, TypeError):
        return False


def run_experiment():
    results = []

    experiments = [
        ("before", i, original_prompts[i - 1], False)
        for i in range(1, 6)
    ] + [
        ("after", review["id"], make_fixed_prompt(review["text"]), True)
        for review in reviews
    ]

    total_runs = len(experiments) * RUNS
    current_run = 0

    for stage, review_id, prompt, fixed in experiments:
        for run_number in range(1, RUNS + 1):
            current_run += 1

            print(
                f"\n[{current_run}/{total_runs}] "
                f"{stage.upper()} | Review {review_id} "
                f"| Run {run_number}"
            )

            try:
                answer = ask_model(prompt, use_fixed_prompt=fixed)
                valid_json = parse_json_response(answer)
                error = None

                print("Valid JSON:", valid_json)
                print("Response:", answer)

            except Exception as exc:
                answer = ""
                valid_json = False
                error = str(exc)
                print("API error:", error)

            results.append({
                "stage": stage,
                "review_id": review_id,
                "run": run_number,
                "model": MODEL,
                "temperature": TEMPERATURE,
                "prompt": prompt,
                "response": answer,
                "valid_json": valid_json,
                "error": error,
            })

            # Keep a small pause between API requests
            time.sleep(0.5)

    with open("W6D5_results.json", "w", encoding="utf-8") as file:
        json.dump(results, file, ensure_ascii=False, indent=2)

    return results


def write_notes(results):
    before = [r for r in results if r["stage"] == "before"]
    after = [r for r in results if r["stage"] == "after"]

    before_valid = sum(r["valid_json"] for r in before)
    after_valid = sum(r["valid_json"] for r in after)

    with open("notes.md", "a", encoding="utf-8") as file:
        file.write("\n\n# W6D5 - Prompt Debugging\n")

        file.write("\n## Criteria for a consistent answer\n")
        file.write(
            "\n1. Every response should use the same output structure.\n"
            "2. The sentiment should use one of three fixed labels: "
            "positive, negative, or mixed.\n"
            "3. The explanation should be short and based on the review.\n"
            "4. Names and products should be listed in a consistent format.\n"
            "5. Missing information should not be invented.\n"
        )

        file.write("\n## Experiment settings\n")
        file.write(f"\n- Model: {MODEL}\n")
        file.write(f"- Temperature: {TEMPERATURE}\n")
        file.write("- Runs per review and version: 3\n")
        file.write("- Total planned responses: 30\n")
        file.write(
            "\nAll original prompt texts were sent unchanged. "
            "The repaired version used a system prompt, context, "
            "instructions, and a fixed JSON output format.\n"
        )

        file.write("\n## JSON parsing comparison\n")
        file.write("\n| Version | Valid JSON | Total responses |\n")
        file.write("|---|---:|---:|\n")
        file.write(f"| Before repair | {before_valid} | {len(before)} |\n")
        file.write(f"| After repair | {after_valid} | {len(after)} |\n")

        file.write("\n## Original responses\n")
        for item in before:
            file.write(
                f"\n### Review {item['review_id']} - "
                f"Run {item['run']}\n"
            )
            file.write(f"\n{item['response'] or item['error']}\n")

        file.write("\n## Repaired prompt responses\n")
        for item in after:
            file.write(
                f"\n### Review {item['review_id']} - "
                f"Run {item['run']}\n"
            )
            file.write(f"\n{item['response'] or item['error']}\n")

        file.write("\n## Problems to investigate in the original prompt\n")
        file.write(
            "\n1. The output format is not specified, so responses "
            "may use paragraphs, bullets, or different layouts.\n"
            "2. 'كويس ولا وحش' does not define fixed sentiment labels, "
            "so the model may use different wording or ratings.\n"
            "3. 'متطولش' is subjective and does not set a clear length "
            "limit for the explanation.\n"
            "4. 'أي أسماء أو منتجات' does not define entity categories "
            "or how to represent missing entities.\n"
            "5. The prompt does not explicitly forbid adding details "
            "that are not present in the review.\n"
        )

        file.write("\n## Did the problems appear in the results?\n")
        file.write(
            "\nUse the recorded responses above to check which issues "
            "actually occurred. Add examples from different runs, such "
            "as differences in labels, explanation length, entity "
            "format, or extra text around JSON. Do not claim a problem "
            "was observed unless the outputs support it.\n"
        )

        file.write("\n## Reflection\n")
        file.write(
            "\n- Did the fixed prompt make the responses more consistent?\n"
            "- Which review was hardest to classify, and why?\n"
            "- Did all repaired responses contain valid JSON?\n"
            "- What would I improve next time?\n"
        )

    print("\nJSON results saved to W6D5_results.json")
    print("Notes appended to notes.md")
    print(f"Before repair: {before_valid}/{len(before)} valid JSON")
    print(f"After repair: {after_valid}/{len(after)} valid JSON")


if __name__ == "__main__":
    if not os.environ.get("GROQ_API_KEY"):
        raise RuntimeError(
            "GROQ_API_KEY is missing. Set it in your terminal first."
        )

    results = run_experiment()
    write_notes(results)